# Model Training & Validation Hyperparameter Selection - Project 1
> **Production Alignment Note:** Mirrors `src/training/train.py`. Splits `train.parquet` into 80% fit and 20% validation, selects `scale_pos_weight` from `(1.0, 2.0, 4.0)` by maximizing validation PR-AUC subject to validation precision `>= 0.95` (selecting `1.0` at `max_depth=6`), and evaluates out-of-sample on `test.parquet`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.ingestion.utils import CANONICAL_FEATURE_COLS
from src.training.train import _select_validation_model

train_df = pd.read_parquet(PROJECT_ROOT / "data/processed/train.parquet")
test_df = pd.read_parquet(PROJECT_ROOT / "data/processed/test.parquet")
X_train_full, y_train_full = train_df[CANONICAL_FEATURE_COLS], train_df["Class"]
X_test, y_test = test_df[CANONICAL_FEATURE_COLS], test_df["Class"]

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, random_state=42, stratify=y_train_full
)

In [ ]:
model, best_params, val_metrics = _select_validation_model(X_fit, y_fit, X_val, y_val)
probs = model.predict_proba(X_test)[:, 1]
preds = (probs >= 0.50).astype(int)
print("Selected Params:", best_params)
print("Validation Metrics:", val_metrics)
print(
    f"Holdout ROC-AUC={roc_auc_score(y_test, probs):.4f} | "
    f"PR-AUC={average_precision_score(y_test, probs):.4f} | "
    f"Recall={recall_score(y_test, preds):.4f} | "
    f"Precision={precision_score(y_test, preds):.4f} | "
    f"F1={f1_score(y_test, preds):.4f}"
)